# Attention (Query, Key, Value)

Attention is the core mechanism of the Transformer — the architecture behind GPT, Claude, LLaMA, and every modern LLM.

It builds on two concepts from earlier notebooks:
- **Dot product** (dot_product.ipynb) — measures similarity between vectors
- **Softmax** (softmax.ipynb) — turns raw scores into weights that sum to 1

## 1. The Intuition — A Library Analogy

Imagine you walk into a library looking for information:

| Attention | Library | Example |
|---|---|---|
| **Query (Q)** | What you're looking for | "I need info about cats" |
| **Key (K)** | Label on each book's spine | "Animals", "Cooking", "History" |
| **Value (V)** | Actual content inside the book | The text, facts, knowledge |

You **match your Query against every Key** (which books are relevant?),
then **read the Values** of the best matches (get the actual content).

That's attention:
```
1. Score each Key against the Query    →  Q · K  (dot product)
2. Normalize scores into weights       →  softmax
3. Read Values weighted by relevance   →  weights × V
```

## 2. Step by Step with Words

Let's trace attention through a real sentence.

We use small 4-dimensional vectors to keep the math visible.
In real models, these would be 768 to 8192 dimensions.

In [ ]:
import math

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def softmax(x):
    max_val = max(x)
    exp_values = [math.exp(val - max_val) for val in x]
    total = sum(exp_values)
    return [val / total for val in exp_values]

# Our sentence
sentence = ["The", "cat", "sat", "on", "the", "mat"]

# Q, K, V vectors for each word (4 dimensions)
# In reality these are learned projections: Q = x @ W_Q, K = x @ W_K, V = x @ W_V
Q = {
    "The": [0.1, 0.2, 0.0, 0.3],
    "cat": [0.8, 0.1, 0.5, 0.2],
    "sat": [0.3, 0.9, 0.4, 0.1],
    "on":  [0.1, 0.3, 0.1, 0.8],
    "the": [0.1, 0.2, 0.0, 0.3],
    "mat": [0.6, 0.1, 0.7, 0.3],
}
K = {
    "The": [0.2, 0.1, 0.1, 0.4],
    "cat": [0.7, 0.2, 0.6, 0.1],
    "sat": [0.2, 0.8, 0.3, 0.2],
    "on":  [0.1, 0.2, 0.1, 0.7],
    "the": [0.2, 0.1, 0.1, 0.4],
    "mat": [0.5, 0.1, 0.8, 0.2],
}
V = {
    "The": [0.1, 0.0, 0.2, 0.1],
    "cat": [0.9, 0.3, 0.1, 0.8],
    "sat": [0.2, 0.8, 0.5, 0.1],
    "on":  [0.1, 0.1, 0.3, 0.2],
    "the": [0.1, 0.0, 0.2, 0.1],
    "mat": [0.7, 0.2, 0.6, 0.9],
}

print("Sentence:", " ".join(sentence))
print(f"Each word has Q, K, V vectors (4 dimensions)")
print("=" * 50)
print(f"{'Word':<6} {'Query':>20} {'Key':>20} {'Value':>20}")
print("-" * 70)
for word in sentence:
    print(f"{word:<6} {str(Q[word]):>20} {str(K[word]):>20} {str(V[word]):>20}")

In [ ]:
# Step 1: Pick a query word — let's see what "sat" attends to

query_word = "sat"
q = Q[query_word]

print(f'Step 1: Compute Q·K scores for "{query_word}"')
print(f"Query vector: {q}")
print("-" * 55)

scores = []
for key_word in sentence:
    k = K[key_word]
    score = dot_product(q, k)
    scores.append(score)
    terms = " + ".join(f"{qi}×{ki}" for qi, ki in zip(q, k))
    print(f"  Q(sat) · K({key_word:<3}) = {terms} = {score:.2f}")

print(f"\n  Raw scores: {[round(s, 2) for s in scores]}")
print(f"  Highest score → 'sat' should pay most attention there")

In [ ]:
# Step 2: Apply softmax to get attention weights

weights = softmax(scores)

print(f'Step 2: Softmax → attention weights for "{query_word}"')
print("-" * 50)
print(f"{'Word':<6} {'Raw score':>10} {'Weight':>10}  Attention")
print("-" * 50)

for word, score, weight in zip(sentence, scores, weights):
    bar = '█' * int(weight * 30)
    print(f"{word:<6} {score:>10.2f} {weight:>10.1%}  {bar}")

print(f"\n  Weights sum to {sum(weights):.2f}")
print(f"  'sat' attends most to the words with highest weights")

In [ ]:
# Step 3: Weighted sum of Values = the output for "sat"

d = len(V[sentence[0]])  # dimension size
output = [0.0] * d

print(f'Step 3: Output = weighted sum of Values')
print("-" * 60)

for word, weight in zip(sentence, weights):
    v = V[word]
    weighted = [weight * vi for vi in v]
    print(f"  {weight:.3f} × V({word:<3}) = {weight:.3f} × {v} = [{', '.join(f'{w:.3f}' for w in weighted)}]")
    for i in range(d):
        output[i] += weighted[i]

print(f"\n  Sum (output for 'sat'): [{', '.join(f'{o:.3f}' for o in output)}]")
print(f"\n  This output is a blend of all Values, weighted by relevance.")
print(f"  Words that scored high in Step 1 contribute more to this output.")

### Putting it all together

```
For each word (as query):
  1. Q · K  →  raw scores (how relevant is each other word?)
  2. softmax →  attention weights (normalized to sum to 1)
  3. weights × V →  output (blend of information from relevant words)
```

The word doesn't just look at itself — it gathers context from the entire sentence.

## 3. Scaled Dot-Product Attention

The full formula from the original Transformer paper:

```
Attention(Q, K, V) = softmax(Q · K^T / √d_k) × V
```

Why divide by `√d_k`? Without scaling, large dimensions make dot products too large,
which pushes softmax into saturation (one weight ≈ 1, rest ≈ 0).

In [ ]:
# Why scaling matters

d_k = len(q)  # dimension = 4
scale = math.sqrt(d_k)

scores_unscaled = scores  # from our earlier computation
scores_scaled = [s / scale for s in scores]

weights_unscaled = softmax(scores_unscaled)
weights_scaled = softmax(scores_scaled)

print(f"Scaling by √d_k = √{d_k} = {scale:.2f}")
print("=" * 60)
print(f"{'Word':<6} {'Unscaled':>10} {'Scaled':>10} | {'W/o scale':>10} {'W/ scale':>10}")
print(f"{'':6} {'score':>10} {'score':>10} | {'weight':>10} {'weight':>10}")
print("-" * 60)

for word, us, ss, wu, ws in zip(sentence, scores_unscaled, scores_scaled,
                                  weights_unscaled, weights_scaled):
    print(f"{word:<6} {us:>10.3f} {ss:>10.3f} | {wu:>10.1%} {ws:>10.1%}")

print(f"\nWith d_k=4 the difference is small.")
print(f"But in real models (d_k=64 to 128), unscaled scores would be")
print(f"~8-11x larger, making softmax almost one-hot (winner takes all).")
print(f"Scaling keeps the distribution smooth so multiple words contribute.")

In [ ]:
# Demonstrate the problem with high dimensions

import random
random.seed(42)

print("Effect of dimension size on dot product magnitude")
print("=" * 55)
print(f"{'d_k':>6} {'Avg dot product':>16} {'√d_k':>6} {'Scaled':>10}")
print("-" * 55)

for d in [4, 16, 64, 128, 512]:
    dots = []
    for _ in range(1000):
        a = [random.gauss(0, 1) for _ in range(d)]
        b = [random.gauss(0, 1) for _ in range(d)]
        dots.append(abs(dot_product(a, b)))
    avg = sum(dots) / len(dots)
    print(f"{d:>6} {avg:>16.2f} {math.sqrt(d):>6.1f} {avg/math.sqrt(d):>10.2f}")

print(f"\nDot products grow with √d_k. Dividing by √d_k keeps them stable.")

## 4. Attention Heatmap

A full attention map shows which words attend to which.
Each row is a Query, each column is a Key.

In [ ]:
# Compute full attention map for the sentence

d_k = 4
scale = math.sqrt(d_k)
shades = " ░▒▓█"

print(f'Attention heatmap: "{" ".join(sentence)}"')
print(f"Each row = Query word, each column = Key word")
print(f"Darker = more attention")
print()

# Header
print(f"{'Q \\ K':<6}", end="")
for word in sentence:
    print(f"{word:>6}", end="")
print()
print("-" * (6 + 6 * len(sentence)))

all_weights = []
for q_word in sentence:
    q = Q[q_word]
    row_scores = [dot_product(q, K[k_word]) / scale for k_word in sentence]
    row_weights = softmax(row_scores)
    all_weights.append(row_weights)
    
    print(f"{q_word:<6}", end="")
    for w in row_weights:
        shade_idx = min(int(w * len(shades) * 2), len(shades) - 1)
        shade = shades[shade_idx]
        print(f"{w:>5.0%}{shade}", end="")
    print()

print(f"\nRead each row: where does that word focus its attention?")
print(f"This is the core computation inside every transformer layer.")

## 5. Multi-Head Attention

One attention head captures one type of relationship.
But language has many types of relationships:

- **Syntactic**: "sat" relates to "cat" (subject-verb)
- **Semantic**: "cat" relates to "mat" (things in a scene)
- **Positional**: "the" relates to "cat" or "mat" (article-noun)

**Multi-head attention** runs multiple attention heads in parallel,
each with its own Q, K, V projections, so each can learn a different pattern.

```
Input → [Head 1] → attention output 1
      → [Head 2] → attention output 2    → Concatenate → Linear → Output
      → [Head 3] → attention output 3
```

In [ ]:
# Simulating 3 attention heads with different patterns

random.seed(123)

def make_random_vectors(words, d):
    return {w: [round(random.uniform(0, 1), 2) for _ in range(d)] for w in words}

def compute_attention_weights(sentence, Q, K, d_k):
    scale = math.sqrt(d_k)
    result = {}
    for q_word in sentence:
        scores = [dot_product(Q[q_word], K[k_word]) / scale for k_word in sentence]
        result[q_word] = softmax(scores)
    return result

heads = []
for h in range(3):
    q_vecs = make_random_vectors(sentence, 3)
    k_vecs = make_random_vectors(sentence, 3)
    weights = compute_attention_weights(sentence, q_vecs, k_vecs, 3)
    heads.append(weights)

# Show what each head focuses on for "sat"
query_word = "sat"
print(f'Multi-Head Attention: what does "{query_word}" attend to?')
print("Each head can learn a different pattern.")
print("=" * 60)

for h_idx, head in enumerate(heads):
    weights = head[query_word]
    top_word = sentence[weights.index(max(weights))]
    print(f"\n  Head {h_idx + 1}: focuses most on '{top_word}'")
    for word, w in zip(sentence, weights):
        bar = '█' * int(w * 25)
        print(f"    {word:<6} {w:>5.1%}  {bar}")

print(f"\nEach head captures different relationships.")
print(f"The outputs are concatenated and mixed through a linear layer.")
print(f"Real models use 8-128 heads per layer.")

In [ ]:
# How multi-head splits the dimensions

print("Multi-Head dimension splitting")
print("=" * 55)
print(f"{'Model':>12} {'d_model':>8} {'Heads':>6} {'d_k per head':>13}")
print("-" * 55)

configs = [
    ("GPT-2 Small", 768, 12),
    ("GPT-2 Large", 1280, 20),
    ("GPT-3", 12288, 96),
    ("LLaMA-7B", 4096, 32),
    ("LLaMA-70B", 8192, 64),
]

for name, d_model, heads in configs:
    d_k = d_model // heads
    print(f"{name:>12} {d_model:>8} {heads:>6} {d_k:>13}")

print(f"\nd_model is split evenly across heads.")
print(f"Each head works with a smaller slice of the full vector.")
print(f"This is why d_k in the scaling formula is per-head, not d_model.")

## 6. Self-Attention vs Cross-Attention

| | Self-Attention | Cross-Attention |
|---|---|---|
| **Q comes from** | Same sequence | Sequence A |
| **K, V come from** | Same sequence | Sequence B |
| **Purpose** | Understand relationships within text | Connect two different inputs |
| **Example** | "The cat sat" understanding itself | Query attending to retrieved documents |

In [ ]:
# Self-attention: Q, K, V all from the same sentence

print("Self-Attention")
print("The model understands relationships WITHIN a sentence.")
print("=" * 50)
print()
print("  Sentence: 'The cat sat on the mat'")
print()
print("  Q, K, V all come from the same sentence:")
print("    Q('sat') · K('The')  → how much should 'sat' look at 'The'?")
print("    Q('sat') · K('cat')  → how much should 'sat' look at 'cat'?")
print("    Q('sat') · K('sat')  → how much should 'sat' look at itself?")
print("    ...")
print()
print("  This is how the model learns that 'sat' is about 'cat',")
print("  not about 'mat' — even though they rhyme.")

In [ ]:
# Cross-attention: Q from one source, K/V from another

print("Cross-Attention (used in RAG)")
print("The model connects the QUERY with RETRIEVED DOCUMENTS.")
print("=" * 60)

query_tokens = ["What", "is", "backpropagation", "?"]
doc_tokens = ["Backprop", "computes", "gradients", "for", "weight", "updates"]

# Fake attention weights for demonstration
cross_weights = {
    "What":              [0.10, 0.05, 0.15, 0.05, 0.10, 0.55],
    "is":                [0.05, 0.40, 0.20, 0.15, 0.10, 0.10],
    "backpropagation":   [0.50, 0.05, 0.25, 0.02, 0.08, 0.10],
    "?":                 [0.15, 0.10, 0.20, 0.05, 0.20, 0.30],
}

print(f"\n  User query:      '{' '.join(query_tokens)}'")
print(f"  Retrieved doc:   '{' '.join(doc_tokens)}'")
print(f"\n  Q comes from the query, K and V come from the document.")
print()

print(f"  {'Q (query)':<20}", end="")
for t in doc_tokens:
    print(f"{t:>10}", end="")
print("  ← K, V (document)")
print(f"  {'-'*80}")

for q_token in query_tokens:
    print(f"  {q_token:<20}", end="")
    for w in cross_weights[q_token]:
        shade = '█' * int(w * 10)
        print(f"{w:>5.0%}{shade:>5}", end="")
    print()

print(f"\n  'backpropagation' strongly attends to 'Backprop' and 'gradients'")
print(f"  This is how the model finds relevant info in retrieved documents.")

## 7. Why This Matters for RAG

In a RAG pipeline:

```
1. User asks a question
2. Retriever finds relevant documents (using dot product / cosine similarity)
3. Documents are concatenated with the question
4. The model uses ATTENTION to focus on the relevant parts
```

Attention is what makes the model actually **use** the retrieved context
instead of just hallucinating an answer.

In [ ]:
# RAG context: the model sees [question + retrieved docs] as one sequence
# Self-attention lets the question tokens attend to the document tokens

print("How attention works in RAG")
print("=" * 60)
print()
print("  Input to the model:")
print("  ┌──────────────────────────────────────────────┐")
print("  │ Context: Backpropagation computes gradients  │ ← retrieved")
print("  │ by propagating error backward through the    │")
print("  │ network layers using the chain rule.         │")
print("  │                                              │")
print("  │ Question: What is backpropagation?           │ ← user query")
print("  └──────────────────────────────────────────────┘")
print()
print("  Self-attention connects them:")
print("    Q('What')            attends to → 'computes', 'gradients'")
print("    Q('backpropagation') attends to → 'Backpropagation', 'chain rule'")
print()
print("  Without attention, the model would ignore the context.")
print("  With attention, it can extract the answer from the retrieved docs.")
print()
print("  This is also why CHUNK SIZE matters in RAG:")
print("  - Too small → not enough context for attention to work with")
print("  - Too large → attention weights spread too thin (diluted)")
print("  - Just right → attention can focus on the relevant sentences")

## 8. Why This Matters for Fine-Tuning

When you fine-tune a model, you're adjusting the weight matrices
that produce Q, K, and V.

```
Q = input × W_Q    ← these weight matrices are what you fine-tune
K = input × W_K
V = input × W_V
```

**LoRA** (Low-Rank Adaptation) specifically targets these projection matrices
because they control **what the model pays attention to**.

In [ ]:
# Why LoRA targets Q, K, V projections

print("What fine-tuning actually changes")
print("=" * 60)
print()
print("  Before fine-tuning (general model):")
print("    W_Q produces queries that look for general patterns")
print("    → 'sat' attends broadly to all words")
print()
print("  After fine-tuning on medical text:")
print("    W_Q produces queries tuned for medical relationships")
print("    → 'prescribed' strongly attends to 'dosage' and 'patient'")
print()
print("  LoRA (efficient fine-tuning):")
print("  ┌─────────────────────────────────────────────────┐")
print("  │  Original W_Q: 4096 × 4096 = 16M parameters    │")
print("  │  LoRA adds:    4096 × 8 + 8 × 4096 = 65K params│")
print("  │                                                 │")
print("  │  W_Q_new = W_Q_original + LoRA_A × LoRA_B      │")
print("  │                                                 │")
print("  │  Only train 65K params instead of 16M!          │")
print("  │  That's 0.4% of the original — 250x cheaper     │")
print("  └─────────────────────────────────────────────────┘")
print()
print("  LoRA targets W_Q, W_K, W_V because small changes to these")
print("  matrices change WHAT the model attends to — which changes")
print("  how it understands and generates text.")

## 9. NumPy Implementation

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    # Stable softmax
    scores = scores - np.max(scores, axis=-1, keepdims=True)
    weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    output = weights @ V
    return output, weights

# 4 tokens, 3-dimensional Q, K, V
np.random.seed(42)
n_tokens = 4
d_k = 3

Q = np.random.randn(n_tokens, d_k).round(2)
K = np.random.randn(n_tokens, d_k).round(2)
V = np.random.randn(n_tokens, d_k).round(2)

output, weights = scaled_dot_product_attention(Q, K, V)

tokens = ["I", "love", "deep", "learning"]

print("Scaled Dot-Product Attention (NumPy)")
print("=" * 50)

print(f"\nAttention weights:")
print(f"{'Q \\ K':<10}", end="")
for t in tokens:
    print(f"{t:>10}", end="")
print()
print("-" * 50)
for i, t in enumerate(tokens):
    print(f"{t:<10}", end="")
    for j in range(n_tokens):
        print(f"{weights[i][j]:>10.3f}", end="")
    print()

print(f"\nOutput (context-aware representations):")
for i, t in enumerate(tokens):
    print(f"  {t:<10} {np.round(output[i], 3)}")

print(f"\nEach output vector is now enriched with context from other tokens.")

## Summary

### The attention formula

```
Attention(Q, K, V) = softmax(Q · K^T / √d_k) × V
```

| Component | What it does | Analogy |
|---|---|---|
| **Q (Query)** | What am I looking for? | Your search query |
| **K (Key)** | What does each token offer? | Book spine labels |
| **V (Value)** | The actual content | Book contents |
| **Q · K^T** | Match query to keys | Checking each book |
| **/ √d_k** | Prevent scores from exploding | Keep search balanced |
| **softmax** | Normalize to weights (sum = 1) | Rank the matches |
| **× V** | Read the best matches | Extract the answer |

### Where it connects

| Context | How attention is used |
|---|---|
| **Self-attention** | Words understand each other within a sentence |
| **Cross-attention** | Query finds relevant parts of retrieved documents |
| **Multi-head** | Multiple heads capture different types of relationships |
| **RAG** | Attention focuses on relevant chunks from retrieval |
| **Fine-tuning / LoRA** | Adjusts W_Q, W_K, W_V to change what the model attends to |